# 全天候资产配置策略 - 分步调试入口

本 notebook 按 `runner.run_backtest` 的编排拆成可独立运行的单元格，每步产出保留在 kernel 内存中。需要重跑某一步时只执行该单元格，避免重复拉取 RQ 数据。

与 `scripts/run_backtest.py` 走同一套项目代码，全部 `from all_weather.* import ...`。

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
# 加载环境变量
load_dotenv()

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from all_weather.config import load_config
from all_weather.data.rq_provider import RQDataProvider
from all_weather.market import to_framework_market_data, to_period_price, calc_returns
from all_weather.flow import build_weight_schedule, make_flow_data_from_weights
from all_weather.backtest import run_portfolio, run_performance
from all_weather.performance import simple_performance_stats, plot_nav_and_drawdown
from all_weather.weights.strategic import risk_contribution

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
print('项目根:', PROJECT_ROOT)

项目根: d:\Program_work\GF_Quants_local\课题研究x1：全天候策略\all_weather_strategy


In [3]:
cfg = load_config(PROJECT_ROOT / 'config' / 'default.yaml')
print(f"策略: {cfg.strategy} | 资产数: {len(cfg.assets)} | 区间: {cfg.start_date} ~ {cfg.end_date_str}")
cfg.asset_df

策略: risk_parity | 资产数: 8 | 区间: 2021-01-01 ~ 2026-07-03


,asset,order_book_id,asset_class,center_weight,tilt_step
0,沪深300,000300.XSHG,权益,0.100,0.050
1,中证500,000905.XSHG,权益,0.100,0.050
2,1-5年国开债,159649.XSHE,债券,0.600,0.050
3,SGE黄金,AU9999.SGEX,黄金,0.050,0.050
4,豆粕,159985.XSHE,商品,0.050,0.050
5,有色金属,512400.XSHG,商品,0.025,0.025
6,能化,159981.XSHE,商品,0.025,0.025
7,中证短融,511360.XSHG,货币,0.050,0.000


In [4]:
# 最贵的一步：拉取 RQ 日频价格。后续单元格复用 daily_price，不必重复执行。
provider = RQDataProvider()
order_book_ids = [a.order_book_id for a in cfg.assets]
daily_price = provider.get_price_panel(order_book_ids, cfg.start_date, cfg.end_date_str, cfg.price_field)
name_map = {a.order_book_id: a.asset for a in cfg.assets}
daily_price = daily_price.rename(columns=name_map).dropna(how='all').ffill()
daily_price.tail()

d:\miniconda3\envs\dolphin\Lib\site-packages\rqdatac\client.py:263: UserWarning: Your account will be expired after  15 days. Please call us at 0755-22676337 to upgrade or purchase or renew your contract.
  warnings.warn("Your account will be expired after  {} days. "


order_book_id,沪深300,中证500,1-5年国开债,SGE黄金,豆粕,有色金属,能化,中证短融
date,,,,,,,,
2026-06-26,4868.2205,8703.5652,110.130,883.70,2.007,1.839,1.403,113.596
2026-06-29,4926.9210,8821.0944,110.143,886.74,2.008,1.864,1.423,113.597
2026-06-30,4979.4328,9031.3788,110.186,879.03,1.985,1.854,1.418,113.610
2026-07-01,4958.9773,9028.9326,110.089,868.80,1.997,1.848,1.386,113.615
2026-07-02,4812.2957,8694.1756,110.142,887.00,2.066,1.880,1.366,113.626


In [5]:
market_data = to_framework_market_data(daily_price, cfg.asset_df)
market_data.head()

,datetime,code,open_price,high_price,low_price,close_price,volume
0,2021-01-04,000300.XSHG,5267.718100,5267.718100,5267.718100,5267.718100,0
1,2021-01-04,000905.XSHG,6482.786800,6482.786800,6482.786800,6482.786800,0
2,2021-01-04,AU9999.SGEX,398.090000,398.090000,398.090000,398.090000,0
3,2021-01-04,159985.XSHE,1.277000,1.277000,1.277000,1.277000,0
4,2021-01-04,512400.XSHG,1.060174,1.060174,1.060174,1.060174,0


In [6]:
monthly_price = to_period_price(daily_price, cfg.rebalance_freq)
monthly_ret = calc_returns(monthly_price)
monthly_ret.tail()

order_book_id,沪深300,中证500,1-5年国开债,SGE黄金,豆粕,有色金属,能化,中证短融
date,,,,,,,,
2026-03-31,-0.055321,-0.120244,0.003743,-0.108551,0.026277,-0.177178,0.345853,0.001565
2026-04-30,0.080282,0.096121,0.002541,-0.005202,0.044444,0.068583,0.002907,0.001298
2026-05-31,0.017643,0.001204,0.003109,-0.028256,-0.039315,-0.072676,-0.081159,0.001067
2026-06-30,0.017847,0.080365,0.001463,-0.107548,-0.044295,-0.056489,-0.105363,0.000749
2026-07-31,-0.033565,-0.037337,-0.000399,0.009067,0.040806,0.014024,-0.036671,0.000141


In [7]:
# 调整策略参数后只需重跑此格 + 后续格，不必重跑取数。
weight_schedule = build_weight_schedule(
    monthly_ret,
    cfg.asset_df,
    strategy=cfg.strategy,
    lookback=cfg.lookback,
    trend_fast=cfg.trend_fast,
    trend_slow=cfg.trend_slow,
    min_weight=cfg.weight_bounds.min,
    max_weight=cfg.weight_bounds.max,
)
weight_schedule.tail().style.format('{:.2%}')

d:\miniconda3\envs\dolphin\Lib\site-packages\pandas\core\frame.py:11238: RuntimeWarning: Degrees of freedom <= 0 for slice
  base_cov = np.cov(mat.T, ddof=ddof)
d:\miniconda3\envs\dolphin\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
d:\miniconda3\envs\dolphin\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


order_book_id,沪深300,中证500,1-5年国开债,SGE黄金,豆粕,有色金属,能化,中证短融
2026-03-31 00:00:00,0.57%,0.44%,16.74%,1.02%,0.71%,0.32%,0.90%,79.30%
2026-04-30 00:00:00,0.56%,0.43%,16.51%,1.03%,0.70%,0.32%,0.90%,79.54%
2026-05-31 00:00:00,0.64%,0.47%,18.58%,1.06%,0.74%,0.34%,0.94%,77.23%
2026-06-30 00:00:00,0.69%,0.55%,20.61%,0.89%,0.82%,0.35%,0.88%,75.21%
2026-07-31 00:00:00,0.74%,0.56%,21.45%,0.94%,1.08%,0.38%,0.96%,73.89%


In [8]:
flow_data = make_flow_data_from_weights(
    weight_schedule,
    market_data,
    cfg.asset_df,
    initial_cash=cfg.initial_cash,
    fee_rate=cfg.trade_fee_rate,
)
flow_data.head(20)

已截断 1 条超出行情范围 [2021-01-04, 2026-07-02] 的调仓信号。
【提示】标的 1-5年国开债 (159649.XSHE) 在 2021-03-01 无有效价格，本次调仓忽略，待数据可用后自动建仓。


,买卖日期,证券代码,买卖数量,买卖权重,买卖价格,买卖收益,保证金比例,买卖方向,盈利方向,交易费用,交易批次
0,2021-01-04,CASH,10000000.0,0.000000,1.000000,0.0,1.0,划入,1,0.000000,0
1,2021-03-01,000300.XSHG,0.0,0.125000,5418.783700,0.0,1.0,买入,1,250.000000,1
2,2021-03-01,000905.XSHG,0.0,0.125000,6488.438800,0.0,1.0,买入,1,250.000000,1
3,2021-03-01,159981.XSHE,0.0,0.125000,1.091000,0.0,1.0,买入,1,250.000000,1
4,2021-03-01,159985.XSHE,0.0,0.125000,1.256000,0.0,1.0,买入,1,250.000000,1
5,2021-03-01,511360.XSHG,0.0,0.125000,101.443000,0.0,1.0,买入,1,250.000000,1
6,2021-03-01,512400.XSHG,0.0,0.125000,1.160025,0.0,1.0,买入,1,250.000000,1
7,2021-03-01,AU9999.SGEX,0.0,0.125000,367.370000,0.0,1.0,买入,1,250.000000,1
8,2021-03-31,511360.XSHG,0.0,0.683081,101.747000,0.0,1.0,买入,1,1366.162793,2
9,2021-03-31,AU9999.SGEX,0.0,0.002118,356.790000,0.0,1.0,买入,1,4.235913,2


In [9]:
# 行情已缓存，重跑回测很快。
results = run_portfolio(
    market_data=market_data,
    flow_data=flow_data,
    position_data=None,
    margin=cfg.backtest.margin,
    allow_loan=cfg.backtest.allow_loan,
    weight_type=cfg.backtest.weight_type,
)
if isinstance(results, tuple):
    backtest_result, position_result, flow_data_new, _ = results
    results_map = {'strategy_0': {'backtest_result': backtest_result, 'position_result': position_result, 'flow_data_new': flow_data_new}}
else:
    results_map = results
backtest_result = next(iter(results_map.values()))['backtest_result']
backtest_result['npv'].tail()


开始回测，共 1330 个交易日
回测进度: [1/1330] 0.1% | 当前日期: 2021-01-04 | 净值: 1.0000 | 权益: 10,000,000.00
回测进度: [50/1330] 3.8% | 当前日期: 2021-03-19 | 净值: 0.9715 | 权益: 9,715,013.65
回测进度: [100/1330] 7.5% | 当前日期: 2021-06-03 | 净值: 1.0219 | 权益: 10,218,691.52
回测进度: [150/1330] 11.3% | 当前日期: 2021-08-13 | 净值: 1.0546 | 权益: 10,545,688.15
回测进度: [200/1330] 15.0% | 当前日期: 2021-11-02 | 净值: 1.0560 | 权益: 10,559,748.69
回测进度: [250/1330] 18.8% | 当前日期: 2022-01-12 | 净值: 1.0505 | 权益: 10,505,305.43
回测进度: [300/1330] 22.6% | 当前日期: 2022-03-30 | 净值: 1.0866 | 权益: 10,866,133.26
回测进度: [350/1330] 26.3% | 当前日期: 2022-06-16 | 净值: 1.1002 | 权益: 11,001,559.60
回测进度: [400/1330] 30.1% | 当前日期: 2022-08-25 | 净值: 1.0688 | 权益: 10,688,092.85
回测进度: [450/1330] 33.8% | 当前日期: 2022-11-11 | 净值: 1.0611 | 权益: 10,610,929.74
【Warning】卖出159649.XSHE@多时无多单持仓，因此不执行卖出。原因可能为现金额不足而未创建多单仓位
回测进度: [500/1330] 37.6% | 当前日期: 2023-01-30 | 净值: 1.1142 | 权益: 11,141,875.24
回测进度: [550/1330] 41.4% | 当前日期: 2023-04-11 | 净值: 1.0996 | 权益: 10,996,412.05
回测进度: [600/1330] 45.1% | 当前日期: 

updatetime
2026-06-26    1.387542
2026-06-29    1.396647
2026-06-30    1.395341
2026-07-01    1.388702
2026-07-02    1.393106
Name: npv, dtype: float64

In [10]:
# 绩效分析：优先 run_performance（需要 plotly/rqdatac），不可用时回退到 simple_performance_stats
# if run_performance is not None:
if run_performance is not None:
    try:
        analysis = run_performance(
            backtest_result,
            label=cfg.strategy,
            benchmark_code=cfg.performance.benchmark_code,
            free_rate=cfg.performance.free_rate,
            plot_periodic=cfg.performance.plot,
            heatmap_periodic=cfg.performance.plot,
            draw_npv=cfg.performance.plot,
            draw_asset=cfg.performance.plot,
            pretty_print_kpi=True,
        )
    except Exception as exc:
        print(f'run_performance 失败，回退: {type(exc).__name__}: {exc}')
        stats = simple_performance_stats(backtest_result)
        print(stats.to_string())
        if cfg.performance.plot:
            plot_nav_and_drawdown(backtest_result, label=cfg.strategy)
else:
    stats = simple_performance_stats(backtest_result)
    print(stats.to_string())
    if cfg.performance.plot:
        plot_nav_and_drawdown(backtest_result, label=cfg.strategy)


【1】关键绩效指标（KPI 汇总）
                         关键绩效指标                         
--------------------------------------------------------
区间: 2021-01-04 ~ 2026-07-02    无风险利率: 2.00%
--------------------------------------------------------
总收益率                                        39.31  %
年化收益率                                       6.48  %
最大回撤率                                      10.97  %
年化波动率                                      10.19  %
胜率                                            33.94  %
Sharpe                                           0.44   
Calmar                                           0.59   
盈亏比                                           4.30   

【2】联动时序图（策略: risk_parity | 基准: 000300.XSHG）



【3】周期收益统计（年度 / 季度 / 月度）



【4】收益热力图


In [11]:
# 最新一期权重 + 风险贡献
latest_weight = weight_schedule.iloc[-1].sort_values(ascending=False)
print('最新一期目标权重:')
print(latest_weight.to_frame('weight').to_string())

cols = latest_weight.index
hist = monthly_ret[cols].tail(cfg.lookback).fillna(0.0)
cov = hist.cov().values + np.eye(len(cols)) * 1e-8
rc = pd.Series(risk_contribution(latest_weight.values, cov), index=cols).sort_values(ascending=False)
print('\n风险贡献:')
print(rc.to_frame('risk_contribution').to_string())

最新一期目标权重:
                 weight
order_book_id          
中证短融           0.738855
1-5年国开债        0.214510
豆粕             0.010835
能化             0.009611
SGE黄金          0.009409
沪深300          0.007385
中证500          0.005634
有色金属           0.003761

风险贡献:
               risk_contribution
order_book_id                   
1-5年国开债                 0.125001
SGE黄金                   0.125000
沪深300                   0.125000
能化                      0.125000
中证500                   0.125000
豆粕                      0.125000
有色金属                    0.125000
中证短融                    0.124999


In [19]:
# ECharts 联动时序图：三图共享同一时间轴指示线
# 依赖 pyecharts；用于替代 Plotly 在 shared_xaxes 下无法稳定显示贯穿三图 spike 的场景。
import importlib
import all_weather.backtest.echarts_viz
importlib.reload(all_weather.backtest.echarts_viz)

from all_weather.backtest.echarts_viz import draw_combined_timeseries_echarts
ECHARTS_OUTPUT = PROJECT_ROOT / "output" / "combined_timeseries_echarts.html"

echarts_path = draw_combined_timeseries_echarts(
    backtest_result,
    label=cfg.strategy,
    benchmark_code=cfg.performance.benchmark_code,
    output_path=ECHARTS_OUTPUT,
    notebook=True,
)

print("ECharts HTML 输出:", echarts_path or ECHARTS_OUTPUT)

[autoreload of all_weather.backtest.echarts_viz failed: Traceback (most recent call last):
  File "d:\miniconda3\envs\dolphin\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\miniconda3\envs\dolphin\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
UnicodeDecodeError: 'gbk' codec can't decode byte 0xaa in position 809: illegal multibyte sequence
]


ECharts HTML 输出: d:\Program_work\GF_Quants_local\课题研究x1：全天候策略\all_weather_strategy\output\combined_timeseries_echarts.html
